In [0]:
%sql 

-- Vista temporal de ventas válidas (no vacías/nulls)

create or replace temp view ventas_no_vacias as 
select *
from fabriperezchauca.bronze.ventas
where coalesce(fecha, cantidad, producto, talla, precio_dado, ajuste_precio,
               precio_original, precio_total, forma_pago, regateo, categoria,
               observaciones) IS NOT NULL

## A. Consistencia de `precio_total`

Observaciones:
- 10 filas donde `precio_total` ≠ `cantidad × precio_dado` (de 2252)
- 2 son redondeo del precio unitario (diferencia de centavos). 8 son errores de digitación, sin forma de saber qué columna está mal
- Diferencia total: ~$50
- **En silver: `precio_total` se recalcula como `cantidad × precio_dado`.** Así la cuenta cuadra siempre, igual que con `ajuste_precio`

In [0]:
%sql

with convertido as (
    select 
        try_cast(cantidad as double) as cantidad,
        try_cast(regexp_replace(precio_dado, '[^0-9.]', '') as double) as precio_dado,
        try_cast(regexp_replace(precio_total, '[^0-9.]', '') as double) as precio_total
    from ventas_no_vacias
)

select 
    cantidad,
    precio_dado,
    precio_total,
    cantidad * precio_dado as total_calculado
from convertido
where abs(precio_total - cantidad * precio_dado) > 0.01

## B. `regateo` contra el descuento

Observaciones:
- Coherentes: 2210 filas (98,1 %). Contradicciones: 26 (`NO` + descuento: 20, `SI` + sin ajuste: 6). Sin dato: 16
- El negocio definió que `regateo` significa solo "hubo descuento": la columna se deduce de los precios
- **En silver: `regateo` no se incluye.** Se deduce de `descuento > 0`. El valor original queda en bronze
- Limitación: en las 6 filas `SI` + sin ajuste, si hubo un regateo real anotado con el mismo precio en ambas columnas, ese descuento se pierde

In [0]:
%sql

with convertido as (
    select
        upper(trim(regateo)) as regateo,
        try_cast(regexp_replace(precio_original, '[^0-9.]', '') as double) as precio_original,
        try_cast(regexp_replace(precio_dado, '[^0-9.]', '') as double) as precio_venta
    from ventas_no_vacias
)

select 
    regateo,
    case 
        when precio_original is null or precio_venta is null then 'sin precio'
        when precio_original > precio_venta then 'descuento'
        when precio_original < precio_venta then 'recargo'
        else 'sin ajuste'
    end as tipo_ajuste,
    count(*) as filas
from convertido
group by all
order by regateo, tipo_ajuste